# Calling C functions from Python

The code is compiled and generates a dynamic link library or DLL.

## Scalar multiplication

### Compilation (scalar multiplication)

In [ ]:
!gcc -fPIC -shared -march=native scalar_multiplication_lib.c -o scalar_multiplication_lib.so

---

In [ ]:
import ctypes
import timeit
import numpy as np

In [ ]:
n = 10
repetitions = 10
T_s = 0
T_numpy = 0
T_avx_python = 0
T_avx_numpy = 0

### Python arrays version ($T_s$)

In [ ]:
def run(n):
	x = 2
	v = [i for i in range(1, n + 1)]

	for i in range(n):
		v[i] = v[i] * x

	# print(v)

times = timeit.repeat(stmt="run(n)", number=repetitions, globals=globals())
average_time = sum(times) / repetitions
T_s = average_time

### NumPy arrays version ($T_{ \text{ numpy } }$)

In [ ]:
def run(n):
	x = 2
	v = np.array(range(1, n + 1))

	for i in range(n):
		v[i] = v[i] * x

	# print(v)

times = timeit.repeat(stmt="run(n)", number=repetitions, globals=globals())
average_time = sum(times) / repetitions
T_numpy = average_time

### Python array with library version ($T_{ \text{ avx }_\text{ python } }$)

In [ ]:
def run(n):
	lib = ctypes.CDLL("./scalar_multiplication_lib.so")
	lib.vector_scalar_multiplication.restype = None
	lib.vector_scalar_multiplication.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_double, ctypes.c_int]

	x = 2
	v = [i for i in range(1, n + 1)]
	c_v = (ctypes.c_double * n)(*v)

	lib.vector_scalar_multiplication(c_v, x, n)

	# print(list(c_v))

times = timeit.repeat(stmt="run(n)", number=repetitions, globals=globals())
average_time = sum(times) / repetitions
T_avx_python = average_time

### NumPy array with library version ($T_{ \text{ avx }_\text{ numpy } }$)

In [ ]:
def run(n):
	lib = ctypes.CDLL("./scalar_multiplication_lib.so")
	lib.vector_scalar_multiplication.restype = None
	lib.vector_scalar_multiplication.argtypes = [
		np.ctypeslib.ndpointer(dtype=np.float64, flags='C_CONTIGUOUS'),
		ctypes.c_double,
		ctypes.c_int
	]

	x = 2
	v = np.array(range(1, n + 1), dtype=np.float64)

	lib.vector_scalar_multiplication(v, x, n - 1)

	# print(v)

times = timeit.repeat(stmt="run(n)", number=repetitions, globals=globals())
average_time = sum(times) / repetitions
T_avx_numpy = average_time

---

In [ ]:
print(f"T_s: {T_s}")
print(f"T_numpy: {T_numpy}")
print(f"T_avx_p: {T_avx_python}")
print(f"T_avx_n: {T_avx_numpy}")

print(f"Speedup_numpy: {T_s / T_numpy}")
print(f"Speedup_avx_p: {T_s / T_avx_python}")
print(f"Speedup_avx_n: {T_s / T_avx_numpy}")

## Dot product

### Compilation (dot product)

In [ ]:
!gcc -fPIC -shared -march=native dot_product_lib.c -o dot_product_lib.so

---

In [ ]:
import ctypes
import timeit
import numpy as np

In [ ]:
n = 100
repetitions = 10
original_version = 0
lib_version1 = 0
lib_version2 = 0

### NumPy arrays version

In [ ]:
def run(n):
	vector_range = range(0, n)
	v1 = np.array(vector_range)
	v2 = np.array(vector_range)

	for i in vector_range:
		v2[i] = 1

	result = 0
	for i in vector_range:
		result += (v1[i] + 1) * v2[i]

	# print(result)

times = timeit.repeat(stmt="run(n)", number=repetitions, globals=globals())
average_time = sum(times) / repetitions
print(f"Average time: {average_time}")
original_version = average_time

### Using `lib.so` with numpy arrays

In [ ]:
def run(n):
	lib = ctypes.CDLL("./dot_product_lib.so")
	lib.vector_dot_product.restype = ctypes.c_double
	lib.vector_dot_product.argtypes = [
		np.ctypeslib.ndpointer(dtype=np.float64, flags='C_CONTIGUOUS'),
		np.ctypeslib.ndpointer(dtype=np.float64, flags='C_CONTIGUOUS'),
		ctypes.c_int
	]

	vector_range = range(0, n)
	v1 = np.array(vector_range, dtype=np.float64)
	v2 = np.array(vector_range, dtype=np.float64)

	for i in vector_range:
		v1[i] = 1
		v2[i] = i + 1

	result = lib.vector_dot_product(v1, v2, n)

	# print(f"Dot product: {result}")

times = timeit.repeat(stmt="run(n)", number=repetitions, globals=globals())
average_time = sum(times) / repetitions
print(f"Average time: {average_time}")
lib_version1 = average_time

### Python arrays with library version

In [ ]:
def run(n):
	lib = ctypes.CDLL("./dot_product_lib.so")
	lib.vector_dot_product.restype = ctypes.c_double
	lib.vector_dot_product.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double), ctypes.c_int]

	v1 = [1 for i in range(n)]
	v2 = [i + 1 for i in range(n)]

	c_v1 = (ctypes.c_double * len(v1))(*v1)
	c_v2 = (ctypes.c_double * len(v2))(*v2)

	result = lib.vector_dot_product(c_v1, c_v2, n)

	# print("Dot product:", result)

times = timeit.repeat(stmt="run(n)", number=repetitions, globals=globals())
average_time = sum(times) / repetitions
print(f"Average time: {average_time}")
lib_version2 = average_time

In [ ]:
speed_up1 = original_version / lib_version1
speed_up2 = original_version / lib_version2

print(f"Lib version 1 SpeedUp: {speed_up1}")
print(f"Lib version 2 SpeedUp: {speed_up2}")